# Day 29: Testing with Assertions and pytest

**Today's question:** how can we be sure our code still works after we change it, without checking everything by
hand every time?

Before running a cell marked 🤔, predict what it will show.

## 1. From print-checks to `assert`

`grades.py` (in this folder) has three small functions: `average`, `letter_grade` and `parse_marks`. Open it in
VS Code and read the docstrings, which are the *specification* we will test against.

Since Day 19 we checked functions like this:

In [ ]:
from grades import average, letter_grade, parse_marks

print(average([70, 80, 90]) == 80)
print(letter_grade(72) == "B")
print(parse_marks("78, 85, 92") == [78, 85, 92])

That works, but you have to **read** every line and spot a `False` among the `True`s. With 50 checks, nobody does.

`assert` turns a check into a statement that must be true. If it is true, **nothing happens**. If it is false,
the program **stops** with an `AssertionError`.

In [ ]:
assert average([70, 80, 90]) == 80
assert letter_grade(72) == "B"
print("all checks passed")

🤔 **Predict:** what happens here?

In [ ]:
assert letter_grade(72) == "A"

You can add a message after a comma. It is shown when the assert fails:

In [ ]:
result = letter_grade(72)
assert result == "A", f"expected A for 72, got {result}"

## 2. Which cases should we test?

Testing one "normal" input is not enough. Bugs hide at the **edges**. For `letter_grade` (A from 80, B from 65,
C from 50, otherwise F; an error outside 0–100):

| Kind | Inputs | Why |
|------|--------|-----|
| **normal** | 72 → `"B"`, 90 → `"A"` | the everyday case |
| **edge** | 80, 79, 65, 64, 50, 49, 0, 100 | exactly on and just below each boundary, plus the extremes |
| **error** | -1, 101 | must raise `ValueError` |

🤔 **Predict:** which edge cases would catch a programmer who wrote `score > 80` instead of `score >= 80`?

In [ ]:
assert letter_grade(80) == "A"
assert letter_grade(79) == "B"
assert letter_grade(50) == "C"
assert letter_grade(49) == "F"
assert letter_grade(0) == "F"
assert letter_grade(100) == "A"
print("edge cases passed")

Only `letter_grade(80)` catches it. Every other input gives the same answer with `>` or `>=`. That is why we
test **exactly on** the boundary.

## 3. Testing that an error is raised: `pytest.raises`

`letter_grade(150)` *should* raise `ValueError`. A plain `assert` can't express "this must crash".
`pytest.raises` can: the indented code **must** raise that error, otherwise the check fails.

In [ ]:
import pytest

with pytest.raises(ValueError):
    letter_grade(150)
print("letter_grade(150) raised ValueError, as it should")

🤔 **Predict:** what happens if the code inside does *not* raise?

In [ ]:
with pytest.raises(ValueError):
    letter_grade(90)

`DID NOT RAISE`: pytest complains that the error we expected never happened.

## 4. Test functions

A **test** is a function whose name starts with `test_` and that contains asserts. It takes no parameters and
returns nothing. Each test checks **one** idea and its name says which.

In [ ]:
def test_average_normal():
    assert average([70, 80, 90]) == 80


def test_average_empty_list_is_zero():
    assert average([]) == 0.0


def test_letter_grade_above_100_raises():
    with pytest.raises(ValueError):
        letter_grade(101)

In [ ]:
test_average_normal()
test_average_empty_list_is_zero()
test_letter_grade_above_100_raises()
print("3 tests passed")

Calling every test by hand doesn't scale. That is pytest's job.

## 5. Running a test file with pytest

`test_grades.py` (in this folder) contains 11 test functions for `grades.py`. Open it in VS Code.
pytest follows two naming rules:

- files named **`test_*.py`** are test files;
- functions named **`test_*`** inside them are tests.

In the terminal, from this folder, you run **`uv run pytest`**. From a notebook, `pytest.main([...])` does the
same thing: the list holds the words you would type after `pytest`.

In [ ]:
result = pytest.main(["-q", "test_grades.py"])

One dot per passing test, then a summary. `-q` means *quiet*. `-v` (*verbose*) prints each test's name:

In [ ]:
result = pytest.main(["-v", "test_grades.py"])

## 6. Reading a failure

To see a failure without breaking our real files, the next cell **writes** a small extra test file. One of its
expectations is wrong. (Normally you would type such a file in VS Code; we write it from Python so the demo can
run anywhere.)

In [ ]:
from pathlib import Path

demo_tests = '''from grades import letter_grade


def test_seventy_nine_is_b():
    assert letter_grade(79) == "B"


def test_eighty_five_is_b():
    assert letter_grade(85) == "B"
'''
Path("test_failure_demo.py").write_text(demo_tests, encoding="utf-8")
print("created test_failure_demo.py")

In [ ]:
result = pytest.main(["-q", "test_failure_demo.py"])

**Read a pytest failure like a traceback:**

1. `.F`: the first test passed, the second **F**ailed.
2. The `>` line is the assert that failed.
3. The `E` lines explain it: `assert 'A' == 'B'`: the function returned `'A'`, the test expected `'B'`.
4. The summary line names the failing test.

🤔 **Who is wrong here, the code or the test?** Check the specification: 80 and above is an A. The **test** is
wrong. When a test fails, first decide which side disagrees with the spec, and fix that side.

We delete the demo file so `uv run pytest` in this folder stays green. `Path.unlink()` deletes a file:

In [ ]:
Path("test_failure_demo.py").unlink()
print(Path("test_failure_demo.py").exists())

## 7. Fixing a bug test-first

A user reports: *"`count_words` says an empty message has 1 word, and `'Hi  Sara'` (two spaces) has 3."*

In [ ]:
def count_words(text):
    """Return the number of words in text."""
    return len(text.split(" "))


print(count_words("Hi there Sara"))

**Step 1: write tests that reproduce the bug** *before* touching the code. They should fail.

In [ ]:
def test_count_words_normal():
    assert count_words("Hi there Sara") == 3


def test_count_words_empty_text():
    assert count_words("") == 0


def test_count_words_double_space():
    assert count_words("Hi  Sara") == 2

In [ ]:
test_count_words_normal()

In [ ]:
test_count_words_empty_text()

**Step 2: fix the code.** `.split()` with no argument splits on any run of spaces and ignores empty pieces.

In [ ]:
def count_words(text):
    """Return the number of words in text."""
    return len(text.split())

**Step 3: run all the tests again.** All silent = all pass. The tests stay, so this bug can never come back
unnoticed.

In [ ]:
test_count_words_normal()
test_count_words_empty_text()
test_count_words_double_space()
print("all count_words tests pass")

## 8. Common mistakes

🤔 **Predict:** does this assert pass?

In [ ]:
assert 0.1 + 0.2 == 0.3

Decimal numbers are stored with tiny rounding errors (`0.1 + 0.2` is `0.30000000000000004`). For money, compare
rounded values; pytest also offers `pytest.approx` for this.

In [ ]:
assert round(0.1 + 0.2, 2) == 0.3
assert 0.1 + 0.2 == pytest.approx(0.3)
print("both pass")

| Mistake | What happens | Fix |
|---------|--------------|-----|
| function named `check_average` | pytest ignores it ("no tests ran") | names must start with `test_` |
| file named `grades_test.py` or `tests.py` | the file is not collected | name it `test_something.py` |
| `assert x = 5` | `SyntaxError` | `==` compares, `=` assigns |
| `print(...)` inside a test instead of `assert` | test always passes | tests must **assert** |
| one huge test checking everything | the first failure hides the rest | one idea per test |
| comparing floats with `==` | fails on rounding noise | `round(...)` or `pytest.approx` |
| test changes a provided data file | other tests and runs break | use small strings and lists; clean up new files |

## Recap

1. `assert condition` is silent when true and raises `AssertionError` when false.
2. Test **normal**, **edge** (exactly on boundaries, empty inputs) and **error** cases.
3. `with pytest.raises(ValueError):` checks that an error *is* raised.
4. pytest runs every `test_*` function in every `test_*.py` file: `uv run pytest` (add `-v` for names).
5. A failure shows the `>` line and the `E` explanation. Decide whether the code or the test is wrong.

**Next time:** Project 03, the Expense Tracker: a program split over several modules, saving to a file, with
its own test files.